In [3]:
import pandas as pd
import numpy as np
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics import masi_distance
from nltk.metrics import jaccard_distance
import ast

In [4]:
df_val=pd.read_csv('labelling_validation.csv', index_col=0)
df_val['Claim_Labels'] = df_val['Claim_Labels'].fillna('')
df_val.rename(columns={'Claim_Labels': 'claim_labels'}, inplace=True)
df_val

,ID,Text,claim_labels
0,0,"So knife crime's going to go up, because the e...","1200, 1211"
1,1,Of course Gordon Brown's right to say there's ...,"100, 107, 1300, 1302, 600, 602, 603, 1200, 1208"
2,2,"You won't admit the truth, will you? The truth...",
3,3,"I tell you how we pay for it. We would, for in...","100, 107,"
4,4,"You just, look, there's no point in speculatin...",
...,...,...,...
364,1426,I think the lady makes a very important point....,"100, 103, 1200, 1207, 1900, 1910, 500, 506"
365,1756,I've met some of the people who have rightly c...,"1200, 1207"
366,2601,"Well, I share the frustration of both our ques...","1900, 1910, 400, 406, 200, 202, 230"
367,3070,You're saying people voted for 10% tariffs on ...,"1000, 1006"


In [5]:
df_val_temp = df_val.copy()
def clean_labels(x):
    if pd.isna(x) or x == "":
        return []
    if isinstance(x, str):
        try:
            splits = x.split(',')
            if 'None' in x:
                return []
            elif '' in splits:                
                splits.remove('')
            elif ' ' in splits:                
                splits.remove(' ')
#             elif x[-1]
            return [int(i.strip()) for i in splits]
        except:
            print(x)
            print(splits)
    return x

df_val_temp['claim_labels'] = df_val_temp['claim_labels'].apply(clean_labels)

In [6]:
df_val_temp.rename(columns={'ID': 'id'}, inplace=True)
df_human = df_val_temp.drop(columns=['Text'])
df_human

,id,claim_labels
0,0,"[1200, 1211]"
1,1,"[100, 107, 1300, 1302, 600, 602, 603, 1200, 1208]"
2,2,[]
3,3,"[100, 107]"
4,4,[]
...,...,...
364,1426,"[100, 103, 1200, 1207, 1900, 1910, 500, 506]"
365,1756,"[1200, 1207]"
366,2601,"[1900, 1910, 400, 406, 200, 202, 230]"
367,3070,"[1000, 1006]"


human df ready

In [14]:
df_llm_1=pd.read_csv('llm_labels_1.csv')
df_llm_1

,id,llm_predicted_codes
0,0,"[1200, 1207, 1209]"
1,1,"[600, 603, 500, 508, 100, 107, 105]"
2,2,[]
3,3,"[100, 107, 105, 500, 508]"
4,4,[]
...,...,...
364,1426,"[500, 506, 1000, 1002, 1006, 1005, 600, 601, 6..."
365,1756,"[200, 207, 600, 601]"
366,2601,[]
367,3070,"[1800, 1803]"


In [16]:
df_llm_1.rename(columns={'llm_predicted_codes': 'claim_labels'}, inplace=True)
df_llm_1

,id,claim_labels
0,0,"[1200, 1207, 1209]"
1,1,"[600, 603, 500, 508, 100, 107, 105]"
2,2,[]
3,3,"[100, 107, 105, 500, 508]"
4,4,[]
...,...,...
364,1426,"[500, 506, 1000, 1002, 1006, 1005, 600, 601, 6..."
365,1756,"[200, 207, 600, 601]"
366,2601,[]
367,3070,"[1800, 1803]"


In [26]:
print(type(df_llm_1['claim_labels'].iloc[0]))

<class 'str'>


In [27]:
df_llm_1['claim_labels'] = df_llm_1['claim_labels'].apply(ast.literal_eval)
df_llm_1

,id,claim_labels
0,0,"[1200, 1207, 1209]"
1,1,"[600, 603, 500, 508, 100, 107, 105]"
2,2,[]
3,3,"[100, 107, 105, 500, 508]"
4,4,[]
...,...,...
364,1426,"[500, 506, 1000, 1002, 1006, 1005, 600, 601, 6..."
365,1756,"[200, 207, 600, 601]"
366,2601,[]
367,3070,"[1800, 1803]"


In [47]:
print(type(df_llm_3['claim_labels'].iloc[0]))

<class 'list'>


getting results for each llm iteration

In [41]:
df_llm_2=pd.read_csv('llm_labels_2.csv')
df_llm_2.rename(columns={'llm_predicted_codes': 'claim_labels'}, inplace=True)
df_llm_2['claim_labels'] = df_llm_2['claim_labels'].apply(ast.literal_eval)
df_llm_2

,id,claim_labels
0,0,"[1201, 1211, 1209]"
1,1,"[600, 603, 500, 508, 101, 105]"
2,2,[]
3,3,"[150, 107, 105, 101]"
4,4,[]
...,...,...
364,1426,"[500, 506, 101, 103, 105, 301, 601, 604]"
365,1756,"[200, 207, 208, 600, 606, 700, 701, 705, 709, ..."
366,2601,"[400, 406, 407, 500, 529, 200, 230, 1000, 2011..."
367,3070,"[1800, 1803]"


In [49]:
df_llm_3 = pd.read_csv('llm_labels_3.csv')
df_llm_3['claim_labels'] = df_llm_3['claim_labels'].apply(ast.literal_eval)
df_llm_3 = df_llm_3.drop(columns=['llm_reasoning'])
df_llm_3

,id,claim_labels
0,0,"[1200, 1209, 1207, 1201]"
1,1,"[600, 603, 500, 508, 1000]"
2,2,[]
3,3,"[107, 105, 101]"
4,4,[]
...,...,...
364,1426,[]
365,1756,[]
366,2601,[]
367,3070,"[1800, 1803]"


In [11]:
df_llm_4 = pd.read_csv('llm_labels_4.csv')
df_llm_4['claim_labels'] = df_llm_4['claim_labels'].apply(ast.literal_eval)
df_llm_4 = df_llm_4.drop(columns=['llm_reasoning'])
df_llm_4

,id,claim_labels
0,0,"[1209, 1211, 1207]"
1,1,"[100, 107, 1300, 1302, 600, 601, 600, 602]"
2,2,"[100, 300, 500, 1300]"
3,3,"[100, 107, 100, 105, 100, 105, 100, 1300, 1302]"
4,4,"[100, 101]"
...,...,...
364,1426,"[500, 502, 100, 105, 500, 502, 1300, 1302]"
365,1756,"[1300, 1307, 2000, 2012, 1300, 1307, 2000, 201..."
366,2601,"[100, 107, 100, 105, 400, 406, 500, 502, 200, ..."
367,3070,"[100, 107]"


In [6]:
df_llm_6 = pd.read_csv('llm_labels_6.csv')
df_llm_6.rename(columns={'llm_predicted_codes': 'claim_labels'}, inplace=True)
df_llm_6['claim_labels'] = df_llm_6['claim_labels'].apply(ast.literal_eval)
df_llm_6

,id,claim_labels
0,0,"[1200, 1207, 1209]"
1,1,"[600, 603, 500, 508, 100, 107, 105]"
2,2,[]
3,3,"[100, 107, 105, 500, 508]"
4,4,[]
...,...,...
364,1426,"[500, 506, 1000, 1002, 1006, 1005, 600, 601, 6..."
365,1756,"[200, 207, 600, 601]"
366,2601,[]
367,3070,"[1800, 1803]"


In [18]:
df_llm_7 = pd.read_csv('llm_labels_7.csv')
df_llm_7['claim_labels'] = df_llm_7['claim_labels'].apply(ast.literal_eval)
df_llm_7

,id,claim_labels
0,0,"[1200, 1209, 1207]"
1,1,"[600, 603, 500, 508, 1000]"
2,2,[]
3,3,"[1500, 107, 105, 101]"
4,4,[]
...,...,...
364,1426,"[500, 506, 507, 508, 101, 103, 105, 101, 103, ..."
365,1756,[]
366,2601,"[400, 406, 408, 100, 101, 103, 107, 108, 200, ..."
367,3070,"[1800, 1803]"


In [5]:
df_llm_6f = pd.read_csv('llm_labels_6_faster.csv')
df_llm_6f['claim_labels'] = df_llm_6f['claim_labels'].apply(ast.literal_eval)
df_llm_6f

,id,claim_labels
0,0,"[1200, 1207, 1209]"
1,1,"[600, 603, 500, 508, 100, 107, 105]"
2,2,[]
3,3,"[100, 107, 105, 500, 508]"
4,4,[]
...,...,...
364,1426,"[500, 506, 1000, 1002, 1006, 1005, 600, 601, 6..."
365,1756,"[200, 207, 600, 601]"
366,2601,[]
367,3070,"[1800, 1803]"


In [15]:
df_llm_8 = pd.read_csv('llm_labels_8.csv')
df_llm_8['claim_labels'] = df_llm_8['claim_labels'].apply(ast.literal_eval)
df_llm_8

,id,supercategories,claim_labels
0,0,[1200],"[1200, 1207]"
1,1,"[100, 600]","[100, 105, 600, 601, 603]"
2,2,[1000],[1000]
3,3,"[100, 200]","[100, 107, 200, 201]"
4,4,[1000],[1000]
...,...,...,...
364,1426,"[500, 1300]","[500, 506, 1300, 1303]"
365,1756,"[200, 600]","[200, 201, 207, 600, 603]"
366,2601,[],[]
367,3070,[1500],"[1500, 1501]"


In [19]:
df_llm_9 = pd.read_csv('llm_labels_9.csv')
df_llm_9['claim_labels'] = df_llm_9['claim_labels'].apply(ast.literal_eval)
df_llm_9

,id,supercategories,claim_labels
0,0,[1200],"[1200, 1207]"
1,1,"[100, 600]","[100, 105, 600, 601, 603, 604, 607]"
2,2,[1000],[1000]
3,3,"[100, 200]","[100, 105, 200, 201]"
4,4,[1000],[1000]
...,...,...,...
364,1426,"[500, 600]","[500, 506, 600, 601, 604]"
365,1756,"[200, 600]","[200, 207, 600, 603]"
366,2601,"[100, 400, 500, 200]","[100, 105, 200, 201, 230, 400, 408, 500, 508]"
367,3070,"[100, 500]","[100, 105, 500, 505]"


In [21]:
df_llm_10 = pd.read_csv('llm_labels_10.csv')
df_llm_10['claim_labels'] = df_llm_10['claim_labels'].apply(ast.literal_eval)
df_llm_10

,id,supercategories,claim_labels
0,0,[1200],"[1200, 1207]"
1,1,"[100, 600]","[100, 105, 600, 601, 604]"
2,2,[],[]
3,3,"[100, 500]","[100, 107, 500, 502]"
4,4,[2000],[2000]
...,...,...,...
364,1426,"[500, 600]","[500, 506, 600, 604]"
365,1756,"[200, 600]","[200, 201, 207, 600, 603]"
366,2601,"[100, 400, 500, 200]","[100, 105, 200, 201, 230, 400, 408, 500, 507]"
367,3070,"[1500, 400]","[400, 401, 1500, 1501]"


In [42]:
df_llm_12 = pd.read_csv('llm_labels_12.csv')
df_llm_12['claim_labels'] = df_llm_12['claim_labels'].apply(ast.literal_eval)
df_llm_12

,id,reasoning,supercategories,claim_labels
0,0,The speaker is discussing the relationship bet...,[1200],"[1200, 1206, 1211]"
1,1,The excerpt discusses the link between poverty...,"[100, 200]","[100, 105, 200, 201]"
2,2,The excerpt does not explicitly mention any sp...,[],[]
3,3,"The excerpt discusses tax policy, specifically...","[100, 200]","[100, 105, 200, 201]"
4,4,The excerpt does not mention any specific poli...,[],[]
...,...,...,...,...
364,1426,The excerpt discusses the importance of provid...,"[500, 600]","[500, 506, 600, 601]"
365,1756,The excerpt discusses the Catholic Church's ha...,"[200, 600]","[200, 600, 607, 1200, 1207]"
366,2601,"The excerpt discusses Brexit, animal welfare, ...","[1900, 400, 500, 700]","[400, 401, 500, 508, 700, 701, 1900]"
367,3070,"The text mentions tariffs, which is related to...","[100, 1800]","[100, 107, 1800, 1803]"


In [47]:
df_llm_13 = pd.read_csv('llm_labels_13.csv')
df_llm_13['claim_labels'] = df_llm_13['claim_labels'].apply(ast.literal_eval)
df_llm_13

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 200]","[100, 107, 200, 201]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 105, 107]"
4,4,<|end_header_id|>\n The excerpt does not cont...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 103, 1300, 1302]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,[200],"[200, 201]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[100, 400, 500, 1900]","[100, 107, 400, 406, 500, 508, 1900, 1901]"
367,3070,<|end_header_id|>\n The excerpt mentions tari...,[],[]


In [5]:
df_llm_14 = pd.read_csv('llm_labels_14.csv')
df_llm_14['claim_labels'] = df_llm_14['claim_labels'].apply(ast.literal_eval)
df_llm_14

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 107, 1300, 1302]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 107]"
4,4,<|end_header_id|>\n The excerpt is a rhetoric...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 103, 1300, 1302]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,"[100, 200]","[100, 105, 200]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[100, 1900]","[100, 108, 1900, 1910]"
367,3070,<|end_header_id|>\n The excerpt mentions a sp...,[100],"[100, 107]"


In [15]:
df_llm_15 = pd.read_csv('llm_labels_15.csv')
df_llm_15['claim_labels'] = df_llm_15['claim_labels'].apply(ast.literal_eval)
df_llm_15

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 107, 1300, 1302]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 107]"
4,4,<|end_header_id|>\n The excerpt is a rhetoric...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 103, 1300, 1302]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,"[100, 200]","[100, 105, 200, 201]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[100, 1900]","[100, 108, 1900, 1901]"
367,3070,<|end_header_id|>\n The excerpt mentions a sp...,[100],"[100, 107]"


In [17]:
df_llm_16 = pd.read_csv('llm_labels_16.csv')
df_llm_16['claim_labels'] = df_llm_16['claim_labels'].apply(ast.literal_eval)
df_llm_16

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300]","[100, 105, 1300, 1302]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 105, 107]"
4,4,<|end_header_id|>\n The excerpt is a statemen...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[100, 1300, 500]","[100, 103, 500, 506, 1300, 1302]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,"[100, 200, 1300]","[100, 105, 200, 201, 1300, 1303]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[100, 400, 1300, 1900]","[100, 105, 400, 401, 1300, 1303, 1900, 1910]"
367,3070,<|end_header_id|>\n The excerpt mentions a sp...,[100],"[100, 107]"


In [21]:
df_llm_18 = pd.read_csv('llm_labels_18.csv')
df_llm_18['claim_labels'] = df_llm_18['claim_labels'].apply(ast.literal_eval)
df_llm_18

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 200]","[100, 107, 200, 201]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 105, 107]"
4,4,<|end_header_id|>\n The excerpt does not cont...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[500, 1300]","[500, 506, 1300, 1302]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,[200],"[200, 201]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[1900, 400]","[400, 408, 1900, 1910]"
367,3070,<|end_header_id|>\n The excerpt mentions tari...,[100],[100]


In [26]:
df_llm_19 = pd.read_csv('llm_labels_19.csv')
df_llm_19['claim_labels'] = df_llm_19['claim_labels'].apply(ast.literal_eval)
df_llm_19

,id,supercategories,claim_labels
0,0,[1200],"[1200, 1206, 1211]"
1,1,"[100, 200]","[100, 107, 200, 201]"
2,2,[],[]
3,3,[100],"[100, 105, 107]"
4,4,[],[]
...,...,...,...
364,1426,"[500, 1300]","[500, 506, 1300, 1302]"
365,1756,[200],"[200, 201]"
366,2601,"[1900, 400]","[400, 408, 1900, 1910]"
367,3070,[],[]


In [27]:
df_llm_20 = pd.read_csv('llm_labels_20.csv')
df_llm_20['claim_labels'] = df_llm_20['claim_labels'].apply(ast.literal_eval)
df_llm_20

,id,reasoning,supercategories,claim_labels
0,0,<|end_header_id|>\n The excerpt discusses the...,[1200],"[1200, 1206, 1211]"
1,1,<|end_header_id|>\n The excerpt discusses the...,"[100, 600]","[100, 107, 600, 602]"
2,2,<|end_header_id|>\n The excerpt appears to be...,[],[]
3,3,<|end_header_id|>\n The excerpt discusses tax...,[100],"[100, 105, 107]"
4,4,<|end_header_id|>\n The excerpt does not cont...,[],[]
...,...,...,...,...
364,1426,<|end_header_id|>\n The excerpt discusses the...,"[500, 1300]","[500, 506, 1300, 1303]"
365,1756,<|end_header_id|>\n The excerpt discusses the...,[200],"[200, 201]"
366,2601,<|end_header_id|>\n The excerpt discusses the...,"[1900, 400]","[400, 406, 1900, 1910]"
367,3070,<|end_header_id|>\n The excerpt mentions tari...,[100],"[100, 107]"


In [7]:
df_llm_21 = pd.read_csv('llm_labels_21.csv')
df_llm_21['claim_labels'] = df_llm_21['claim_labels'].apply(ast.literal_eval)
df_llm_21

,id,reasoning,supercategories,claim_labels
0,0,The excerpt discusses the relationship between...,[1200],"[1200, 1211]"
1,1,The text discusses the link between poverty an...,"[100, 1300]","[100, 105, 1300, 1302]"
2,2,The excerpt contains no explicit mention of po...,[],[]
3,3,The text discusses closing tax loopholes that ...,[100],"[100, 107]"
4,4,The text does not contain any explicit mention...,[],[]
...,...,...,...,...
364,1426,The text discusses the issue of youth unemploy...,"[100, 1300]","[100, 103, 1300, 1302]"
365,1756,The text discusses the Catholic Church's handl...,"[200, 1900]","[200, 201, 1900, 1926]"
366,2601,"The excerpt discusses Brexit, its potential be...","[1900, 100]","[100, 107, 1400, 1401, 1900, 1910]"
367,3070,"The excerpt discusses tariffs on cars, which i...",[100],"[100, 107]"


In [17]:
df_llm_22 = pd.read_csv('llm_labels_22.csv')
df_llm_22['claim_labels'] = df_llm_22['claim_labels'].apply(ast.literal_eval)
df_llm_22

,id,reasoning,supercategories,claim_labels
0,0,100 (Macroeconomics) is implied as the discuss...,"[100, 200]","[100, 107, 200, 201]"
1,1,The excerpt discusses the link between poverty...,"[100, 1300, 600]","[100, 107, 600, 602, 1300, 1302]"
2,2,100 (Macroeconomics) is inferred from the ment...,"[100, 200]","[100, 107, 200, 201]"
3,3,100 is chosen because the text discusses taxat...,"[100, 200]","[100, 107, 200, 201]"
4,4,100 (Macroeconomics) is inferred as the superc...,[100],"[100, 107]"
...,...,...,...,...
364,1426,"The excerpt discusses youth unemployment, bene...","[500, 1300, 1900]","[500, 506, 1300, 1302, 1900]"
365,1756,100 (Macroeconomics) is implied as the speaker...,"[100, 200]","[100, 107, 200, 201]"
366,2601,100 (Macroeconomics) is implied by the mention...,"[100, 1300, 1900]","[100, 1300, 1900]"
367,3070,100 (Macroeconomics) is used because the text ...,[100],"[100, 107]"


In [19]:
df_llm_23 = pd.read_csv('llm_labels_23.csv')
df_llm_23['claim_labels'] = df_llm_23['claim_labels'].apply(ast.literal_eval)
df_llm_23

,id,reasoning,supercategories,claim_labels
0,0,The excerpt discusses the relationship between...,[1200],"[1200, 1209]"
1,1,The excerpt discusses the link between poverty...,"[100, 200, 300, 600]","[100, 107, 200, 201, 300, 301, 600, 602]"
2,2,The excerpt does not contain any explicit refe...,[],[]
3,3,"The excerpt discusses tax policies, specifical...",[100],"[100, 107]"
4,4,The excerpt discusses a decision that is far f...,[1000],"[1000, 1001]"
...,...,...,...,...
364,1426,"The excerpt discusses youth unemployment, bene...","[500, 600]","[500, 506, 600, 602]"
365,1756,The excerpt discusses the impact of past abuse...,"[1200, 1900]","[1200, 1207, 1900]"
366,2601,The excerpt discusses the advantages of Brexit...,"[100, 200, 400, 500, 600, 700, 800, 1000, 1200...","[100, 107, 200, 400, 406, 500, 508, 600, 700, ..."
367,3070,1800 is relevant as it discusses trade policie...,[1800],[1800]


llm df ready

----

Inter-rater agreement metric: Krippendorff's Alpha

In [9]:
def prep_for_alpha(df, rater_name):
    df['claim_labels'] = df['claim_labels'].apply(lambda x: frozenset(x))
    return [(rater_name, row['id'], row['claim_labels']) for _, row in df.iterrows()]

In [12]:
# prep human df once
prepped_human_data = prep_for_alpha(df_human, 'human')

In [10]:
def safe_masi_distance(label1, label2):    
    if not label1 and not label2:
        return 0.0
    if not label1 or not label2:
        return 1.0
    return masi_distance(label1, label2)

In [15]:
def safe_jaccard_distance(label1, label2):    
    if not label1 and not label2:
        return 0.0
    if not label1 or not label2:
        return 1.0
    return jaccard_distance(label1, label2)

In [39]:
# RESULT 1
# create combined dataset (repeat for each llm result)
task_data = prepped_human_data + prep_for_alpha(df_llm_1, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2104


In [43]:
# RESULT 2
task_data = prepped_human_data + prep_for_alpha(df_llm_2, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2206


In [50]:
# RESULT 3
task_data = prepped_human_data + prep_for_alpha(df_llm_3, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.1346


In [16]:
# RESULT 4
task_data = prepped_human_data + prep_for_alpha(df_llm_4, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.1578


In [11]:
# RESULT 6
task_data = prepped_human_data + prep_for_alpha(df_llm_6, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2104


In [9]:
# RESULT 6 ('faster')
task_data = prepped_human_data + prep_for_alpha(df_llm_6f, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2109


In [19]:
# RESULT 7
task_data = prepped_human_data + prep_for_alpha(df_llm_7, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2061


In [16]:
# RESULT 8
task_data = prepped_human_data + prep_for_alpha(df_llm_8, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.1835


In [20]:
# RESULT 9
task_data = prepped_human_data + prep_for_alpha(df_llm_9, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.1781


In [22]:
# RESULT 10
task_data = prepped_human_data + prep_for_alpha(df_llm_10, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2469


In [41]:
# JACCARD
# create combined dataset (repeat for each llm result)
task_data = prepped_human_data + prep_for_alpha(df_llm_10, 'llm')
task = AnnotationTask(data=task_data, distance=safe_jaccard_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3378


In [45]:
# RESULT 12
task_data = prepped_human_data + prep_for_alpha(df_llm_12, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.2953


In [50]:
# RESULT 13
task_data = prepped_human_data + prep_for_alpha(df_llm_13, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3153


In [13]:
# RESULT 14
task_data = prepped_human_data + prep_for_alpha(df_llm_14, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3148


In [16]:
# RESULT 15
task_data = prepped_human_data + prep_for_alpha(df_llm_15, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3076


In [18]:
# RESULT 16
task_data = prepped_human_data + prep_for_alpha(df_llm_16, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3095


In [22]:
# RESULT 17
task_data = prepped_human_data + prep_for_alpha(df_llm_18, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3463


In [25]:
# RESULT 19
task_data = prepped_human_data + prep_for_alpha(df_llm_19, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3436


In [28]:
# RESULT 20
task_data = prepped_human_data + prep_for_alpha(df_llm_20, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3599


In [13]:
# RESULT 21
task_data = prepped_human_data + prep_for_alpha(df_llm_21, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3985


In [16]:
# RESULT 21 (JACCARD)
task_data = prepped_human_data + prep_for_alpha(df_llm_21, 'llm')
task = AnnotationTask(data=task_data, distance=safe_jaccard_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.4970


In [18]:
# RESULT 22
task_data = prepped_human_data + prep_for_alpha(df_llm_22, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.1449


In [22]:
# RESULT 23
task_data = prepped_human_data + prep_for_alpha(df_llm_23, 'llm')
task = AnnotationTask(data=task_data, distance=safe_masi_distance)
print(f"Krippendorff's Alpha: {task.alpha():.4f}")

Krippendorff's Alpha: 0.3505
